# Практикум 03 · Маски, контуры и рисование

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

Стандартный **Run All** использует учебные изображения или синтетические данные. Подготовьте Python с OpenCV и NumPy по стартовой инструкции.


# 03. Маски, границы и подписи на изображении

**Тема Yahboom:** 9.3 Image processing and draw. **Время:** 60–90 минут.

## Цель и место выполнения

Создайте изображения, которые отвечают на простые вопросы: где находится красный цвет, какие пиксели достаточно яркие, где проходят резкие границы? Добавьте поверх кадра геометрические фигуры и текст. Это подготовка к выделению объекта, который позже будет попадать в кадр камеры ROSMASTER.

Работайте из корня курса на ноутбуке или бортовом компьютере. Робот остаётся неподвижным. Нужны `scene.png` и `noisy.png` из генератора первого урока. Открывать камеру, подключать ROS и запускать управление движением здесь не требуется.

## Пять понятий

**Бинарная маска** — изображение из нулей и 255. Белый пиксель означает выполнение заданного условия, чёрный — его отсутствие. Значение 255 не показывает силу уверенности: для используемых здесь операций решение имеет только два состояния.

**Порог яркости** сравнивает серый пиксель с одним числом. При `THRESH_BINARY` пиксели строго выше порога становятся белыми. Освещение влияет на результат: один и тот же предмет в тени может перестать проходить порог.

**HSV** описывает цвет через тон H, насыщенность S и яркость V. В 8-битном представлении OpenCV H занимает диапазон 0–179, S и V — 0–255. Красные оттенки находятся около обоих концов шкалы H. Поэтому код объединяет диапазоны 0–10 и 170–179; низкая насыщенность исключает серые и белые области.

**Морфология** изменяет форму белых областей с помощью небольшого ядра. Открытие помогает убрать мелкие отдельные точки, закрытие — заполнить небольшие разрывы. Слишком большое ядро способно уничтожить полезную мелкую деталь. Размер ядра в программе нечётный.

**Границы и аннотации** решают разные задачи. Canny выделяет резкие изменения яркости, используя два порога. Функции рисования добавляют линии, рамки, круги, эллипсы, многоугольники и надписи к копии изображения. Нарисованная рамка сама по себе не является результатом обнаружения объекта.

## Практика: семь шагов

1. Посмотрите параметры и выполните стандартную обработку:

   ```bash
   python3 -m rosmaster_cv.lesson03_drawing --help
   python3 -m rosmaster_cv.lesson03_drawing --input data/scene.png --out outputs/03
   ```

2. Откройте `annotated.png`. Найдите все виды рисунков. Сравните с `data/scene.png`: исходник сохранён, аннотации находятся в отдельном файле. Надписи сделаны латиницей, поскольку стандартный шрифт `cv2.putText` не поддерживает обычный вывод кириллицы.

3. Сравните `threshold.png` и `red_mask.png`. На первом изображении белыми становятся достаточно яркие места, на втором — места, удовлетворяющие цветовому условию. Найдите минимум две области, где эти решения различаются. Проверьте, что обе большие красные фигуры присутствуют в цветовой маске.

4. Измените только порог яркости:

   ```bash
   python3 -m rosmaster_cv.lesson03_drawing --threshold 220 --out outputs/03_bright
   ```

   До запуска предскажите, станет ли белых пикселей больше или меньше. Проверьте `threshold_white_pixels` в двух файлах `report.json`; остальные параметры должны оставаться одинаковыми.

5. Исследуйте порог насыщенности и ширину тона:

   ```bash
   python3 -m rosmaster_cv.lesson03_drawing --s-min 230 --hue-width 3 --out outputs/03_strict
   ```

   Сравните маску со стандартной. По сцене известно, что большие красные фигуры созданы с насыщенностью 220: строгий порог может исключить обе. Объясните результат, а затем подберите параметры, возвращающие две фигуры.

6. Обработайте сцену с шумом двумя размерами ядра:

   ```bash
   python3 -m rosmaster_cv.lesson03_drawing --input data/noisy.png --kernel 3 --out outputs/03_noise3
   python3 -m rosmaster_cv.lesson03_drawing --input data/noisy.png --kernel 9 --out outputs/03_noise9
   ```

   Сравните `red_mask.png`, `opened.png` и `closed.png`. Проверьте, не изменились ли полезные границы. В отчёте записаны количества белых пикселей до и после морфологии; меньшее количество не всегда означает лучшее качество.

7. Измените чувствительность Canny:

   ```bash
   python3 -m rosmaster_cv.lesson03_drawing --canny 20 60 --out outputs/03_edges
   ```

   Сопоставьте количество `edge_pixels` со стандартными порогами 60 и 160. Найдите границы, появившиеся дополнительно, и оцените, полезны ли они для поиска крупных фигур.

## Ожидаемые файлы

В каждом каталоге результатов появятся `annotated.png`, `hsv_h.png`, `hsv_s.png`, `hsv_v.png`, `red_mask.png`, `threshold.png`, `opened.png`, `closed.png`, `canny.png` и `report.json`. Файлы H, S и V показывают отдельные числовые каналы как серые изображения. Канал H хранит значения до 179, поэтому его изображение не использует весь диапазон яркости 0–255.

## Самостоятельное изменение

Сфотографируйте красный предмет на светлом фоне и сохраните `data/my_red_object.jpg`. Сделайте второй кадр при другом освещении. Подберите `--s-min` и `--v-min` на первом кадре; проверьте без изменений на втором. Запишите пропуски и лишние области. Затем в своей копии `lesson03_drawing.py` измените цвет окружности и строку надписи, снова запустите модуль и покажите изменённые строки. Не меняйте одновременно все параметры: так легче объяснить причину результата.

## Три вопроса

1. Зачем красной маске два диапазона H?
2. Почему яркая белая область может попасть в пороговую маску и отсутствовать в красной?
3. Как слишком большое ядро повлияет на тонкую красную линию?

## Что сдать и ошибки

Сдайте сравнение масок для двух освещений, таблицу параметров, результаты двух размеров ядра, свою аннотацию и ответы. В выводе укажите конкретный случай, когда алгоритм ошибается.

**Маска полностью чёрная:** проверьте S/V и наличие нужного цвета в кадре. **Ошибки ядра или Canny:** размер ядра должен быть нечётным от 1 до 31; нижний порог Canny должен быть меньше верхнего.

## Источник и далее

[Yahboom 9.3](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8144/). HSV и морфология добавлены как подготовка к последующим практическим заданиям.

Далее: [04 — Улучшение и восстановление](../lessons/04_enhancement.md).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


In [ ]:
run_tool("generate_data.py")
run_module("lesson03_drawing", "--input", "data/scene.png", "--out", "outputs/03")


In [ ]:
def show_results(relative_directory, limit=4):
    directory = ROOT / relative_directory
    paths = sorted(directory.glob("*.png"))[:limit]
    if not paths:
        print("Изображений пока нет:", directory)
        return
    try:
        from IPython.display import Image, display
    except ImportError:
        for path in paths:
            print(path)
        return
    for path in paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))

show_results("outputs/03")


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
